# 03 · Feature Checks: the gate between the candidates and the model

**What it does.** Screens every candidate for correctness and data quality before any model sees
it, never for relevance: a weak feature passes, a feature that cannot be trusted as it is gets a
flag. It reads EDA's artifacts only: no Spark job, no new pass over the data.

**Five steps, in this order:** `quality` (empty, constant or unreadable columns), `leakage` (the
target under another name), `time_alignment` (a value not known at its period), `stability` (a
relation or distribution that moves over time) and `redundancy` (near-duplicates). Each step
raises **findings** of two kinds:
- a **flag**, "should not enter a model as it is": the feature is **held** until a person records
  `keep` or `exclude`. **For Training, a held feature acts as an excluded feature without a final
  decision: the model is built without it until a person keeps or excludes it.**
- a **note**, "look at this": it never affects training.

Nothing is dropped: a held feature stays a candidate, in the feature ledger and in every report.

**Automation levels** (`automation_level` in the config; this example is `assisted`): the checks
and thresholds are the same at both. At `assisted` you review the flags here and record your
decisions (step 9) before training. At `autonomous` Training starts at once without the held
features; come here to bring a held feature back. At both levels a person signs off every feature
of the model in one gate after training.

**The population** is every eligible client of the observation window, labels included, before
the train/test split. Your decisions here have therefore seen the labels of the clients that
later form the test split; the model results report states it.

**Before you start.** Run `02_eda` first (step 1 at least): Feature Checks reads its artifacts for
the same data definition. For the other example, change `CONFIG` in the setup cell. The CLI:
`python -m core_modelling run conf/base/cc_acquisition.yml --stages eda feature_checks`.

**Docs:** `docs/components/feature_checks.md` · `docs/components/feature_ledger.md` · next:
`04_training`.

In [ ]:
from pathlib import Path

from IPython.display import HTML, display
from pyspark.sql import functions as F

import core_modelling
from core_modelling.artifacts import list_data_definitions
from core_modelling.config import load_config
from core_modelling.config.audit_log import record_decision, resolve_config, show_decisions
from core_modelling.dataset import build_frames
from core_modelling.environment import get_spark_session, load_environment
from core_modelling.feature_checks import run_feature_checks

# The use case and where the framework runs, as absolute paths. The repo's examples are found
# from the installed package; a use-case repo writes its own config path here and may read the
# environment file from $CORE_MODELLING_ENVIRONMENT instead.
REPO = Path(core_modelling.__file__).resolve().parents[2]
CONFIG = REPO / "conf/base/cc_acquisition.yml"  # or conf/base/td_acquisition.yml
ENTITY = "entity_a"
ENVIRONMENT = str(REPO / "conf/local/environment.yml")  # or None: $CORE_MODELLING_ENVIRONMENT

env = load_environment(ENVIRONMENT)
spark = get_spark_session(env)  # Feature Checks never uses it; step 1's data cell does
cfg = load_config(CONFIG)
print(cfg.use_case, cfg.task, cfg.automation_level)
print(cfg.feature_checks)

## 1. Run Feature Checks
`run_feature_checks` applies the decision audit log to the config, reads EDA's artifacts, runs the
five steps and records the flags; `result.views` holds the report's tables and charts, and
`result.config` the config after the run, whose `features.held` lists the held features. It writes:
- **in the audit log**: one `flag` entry per finding whose features changed since the last run (an
  empty list closes the finding), then one `run` entry listing the compared periods. A rerun on
  unchanged data writes only the `run` entry.
- under `<roots.data>/<use case>/04_feature/<data-definition hash>/`: `feature_checks/`
  (`report.html`, `feature_checks.json`, `tables/`, `resolved_config.yml`) and, one folder up,
  Feature Checks' columns in the feature ledger (`ledger.parquet`).

**When it fails**, the message says what is missing: no EDA for this data definition, EDA ran
with other settings (`features.kinds`, `strata`, `eda`, `time.validation_period`, `decision`,
`evaluation.slices`), the environment file maps one of the sources elsewhere since EDA ran, or
the ledger comes from another EDA run than `eda/`. Rerun `02_eda` step 1,
then this cell; a failed run leaves the previous `feature_checks/` in place. The table under the
cell lists the use case's data-definition folders, the one this config uses first (`in_use`), as
in `02_eda` step 1.

**The data, to check a finding on the spot.** The second cell loads the rows the run judged, for
the config after the run: `snapshot`, the snapshot period's rows in pandas (every row after the
analysis filters, with `eligible` and `target`), and `rows`, the eligible rows of every compared
period in Spark (`unit_id`, `period`, the candidates, and `target` in the window's periods only).
Step 8 shows three checks on them; any pandas or Spark query works too. The cell collects the
snapshot again, as EDA does: skip it if you only read the views.

In [ ]:
result = run_feature_checks(cfg, ENTITY, env, spark)
views = result.views
list_data_definitions(env, cfg, ENTITY)

In [ ]:
frames = build_frames(result.config, ENTITY, env, spark, "eda")
snapshot, rows = frames.snapshot, frames.rows

## 2. The choices and the summary
**How to read** the words every later view uses:
- **Concordance**: 0.5 is chance, 1 a perfect ordering. `concordance_window` is the plain
  concordance of the window's periods pooled, a numeric's values in their order (EDA's
  `concordance` is the snapshot's, in the best order when a feature is non-monotone). Per period
  and lagged it is read in the window's direction, so a flipped feature reads below 0.5.
- **Flag** and **note**: a flag holds the feature out of training at both levels until a person
  records `keep` or `exclude`; a note never affects training.
- **Grey**: little support, still shown and still tested: a window period with fewer than
  `strata.min_events` effective events is greyed.
- **Support and significance**: every check runs, whatever the support. A finding with a standard
  error ends its `detail` with its support (effective events) and SE, and says "not significant at
  this support" when the value is within 2 SE of its threshold: such a flag may be noise, so look
  at the data (step 8) before you decide.
- **`null`**: the clients without a value, its own bin in every statistic (PSI, new levels),
  always last.

**The views:**
- **Choices**: what was set before any number. Check the level, the support rule (`min_events`),
  each step's thresholds and the **compared periods**: the window's periods (labelled), then the
  gap periods and the latest period, compared without labels. The validation period is never
  read, and a period a run has compared can no longer become the validation period.
- **EDA headline** and **warnings**: EDA's numbers for the population judged; one warning is always
  there, that the population holds the clients of the later test split.
- **Flags and notes**: per finding, how many features raised it; a flag's features are named, a
  note's only counted (they run to dozens: see its step below).
- **Held features**: every feature with an open flag, with its open flags, `flag_values` (each
  open flag's statistic and this run's value; `not raised`: open from an earlier run),
  `concordance_window` and `missing_share_window`. This is the list Training leaves out.

**Decide:** a threshold that does not fit the use case is a `set` of its leaf, such as
`feature_checks.leakage.concordance_max` (as in `02_eda` step 9), then rerun step 1; it applies at
both levels.

In [ ]:
shown = [views["choices"], views["headline"], views["warnings"], views["raised"], *views["held"]]
display(*[HTML(block) if isinstance(block, str) else block for block in shown])

## 3. Quality
The thresholds below are the defaults; the Choices table shows this run's.

**Each step shows** the findings it raised with their texts, then one table: a row per feature
and finding, its `value` against the `threshold` it reached, the ledger's statistics beside it and
the `detail` last. A last table lists what the step could not compute (no value can exist,
such as no events): no finding either way. A share's `value` and `threshold` read as fractions
(0.95 is 95%), the ledger's share columns beside them as percentages; all are over the window's
eligible rows.

The first four flags are exclusive, in this order; null counts as a value:
- `all_null` (every value null) and `constant` (one value only): usually a column the eligibility
  fixes (a card-acquisition population holds no card, so every card balance is 0) or a dead feed.
  Exclude it.
- `high_missing`: `missing_share_window` ≥ 0.95. Keep it only if the few filled values will be
  filled at scoring too.
- `near_constant`: one value on ≥ 99% of rows (`dominant_share_window`). The `detail` compares the
  rare rows' rate with the dominant value's (Cohen's h, step 4): a rare flag with a clear effect
  can be worth a keep.
- `high_cardinality`: a categorical with over 50 `levels`. Meaningful levels (a nationality) are a
  keep; codes or free text are an exclude, or a grouping built upstream.
- `nulls_may_be_zeros` (note): ≥ 20% null and almost never a zero (`detail`). If a null means 0
  (no transactions), have it filled upstream; the framework never fills a null. A null that means
  "no such product" (a loan's maturity without a loan) is right as it is.
- `nulls_informative` (note): whether the value is filled orders clients (`value`: the null
  concordance, 0.55 to 0.70). The nulls stay their own group. The `detail` names a **null set**,
  features null on the same rows: often a population, such as clients too new for a six-month
  roll-up. A population that behaves differently may deserve an eligibility filter (a minimum
  relationship length) or a model of its own; `02_eda`'s pivots show how it differs.

**Decide:** each flag is a `keep` or an `exclude` (step 9); a note needs no entry.

In [ ]:
shown = views["steps"]["quality"]
display(*[HTML(block) if isinstance(block, str) else block for block in shown])

## 4. Leakage
A leak knows the outcome: the target under another name, or a field filled because of it. It
orders clients far better than any honest feature.
- `too_strong`: the window concordance ≥ 0.80, the plain one or, if higher, the best-order one
  (the values ordered by their rate, so a U shape counts; the `detail` says when it fired). Ask
  how and when the field is filled; unless it is known before the target window starts, exclude
  it.
- `missingness_leak`: whether the value is filled alone orders clients that well (`value`: the null
  concordance, ≥ 0.70). Typical: a field filled only for clients who received an offer or
  applied.
- `rare_value_leak`: beside one value holding ≥ 90% of the filled rows, the other values' rate is
  far above its: Cohen's h ≥ 0.8. h compares two rates on a scale where a move near 0 counts
  less, so 0.2% → 1% is small (0.11) and 5% → 25% is not (0.60); 0.2 is small, 0.5 medium, 0.8
  large. An amount target uses Cohen's d, the difference of the means over their standard
  deviation, on the same bands. `value` is h (d); the `detail` names the dominant value, the
  minority's share and both rates (means). A code the process sets after the outcome looks like
  this, and so does a small real group with high take-up (a premium segment): keep the second
  kind once.

**Decide:** a leak is an `exclude`; a `keep` needs a reason you can defend at the sign-off (when
and how the field is filled).

In [ ]:
shown = views["steps"]["leakage"]
display(*[HTML(block) if isinstance(block, str) else block for block in shown])

## 5. Time alignment
A feature must be known at its period: a value stamped later, or one date's value copied onto
every period, makes the model look better than it will be. Both checks look only at features with
`concordance_window` ≥ 0.6.
- `lagged_signal_drop`: read one period earlier (`lagged_concordance`), the feature keeps less than
  half its signal. `value` is `signal_kept` = (lagged − 0.5) / (window − 0.5): next month's value
  keeps nothing, a slow honest feature nearly all, and below 0 the lagged value orders clients
  the other way. The `detail` is the share of rows without a lagged value (a new client or a
  null), and says when the lagged value orders clients the other way beyond noise. Mostly an
  exclude; a fast-moving honest feature can be kept.
- `client_constant`: the value never changes within a client across the window (`value`:
  `constant_share` ≥ 0.99) and it predicts (`concordance_window` ≥ 0.6). A static attribute known
  at the period (an onboarding channel) is fine; one date's value stamped onto every period is a
  look-ahead. No number tells them apart, only how the column is built: step 8 shows a few
  clients' values across the periods. One `keep` naming the static ones settles them once per
  use case.

**Decide:** `keep` what you know is set at or before its period; `exclude` the rest.

In [ ]:
shown = views["steps"]["time_alignment"]
display(*[HTML(block) if isinstance(block, str) else block for block in shown])

## 6. Stability
- `concordance_unstable` (flag): the concordance differs across the window's periods more than
  chance allows (Cochran's Q, `heterogeneity_p` ≤ 0.01) and by at least half the feature's signal
  (`concordance_range` ≥ 0.5 × |`concordance_window` − 0.5|), so a wobble that is small for a
  strong feature does not fire, and never by less than 0.05, so a feature with no signal needs a
  real swing; the `detail` names the periods tested. A signal present in one period only, or a flipped
  direction, will not hold in production: find the cause before a keep.
- `psi_window`: the distribution moved between the window's first and last periods;
  `psi_latest`: between the window's last period and the latest period, the data production will
  score. From PSI 0.10 a note (`psi_window`, `psi_latest`), from 0.25 a flag
  (`psi_window_large`, `psi_latest_large`; the leaves `feature_checks.stability.psi_note_min` and
  `psi_max`). PSI 0 is no change, below 0.10 usually stable, above 0.25 a large shift. A PSI far
  above 1 means the field is filled differently there (null, say): ask why.
- `new_levels` (note): levels never seen in the window cover ≥ 1% of the latest period's rows (a
  new code, or a field that stopped being filled).
- **The chart**: the concordance per window period of the features `concordance_unstable`
  raised, in the window's direction, so a flip reads below 0.5 (EDA's report shows every
  candidate's).

**Decide:** a flag is a `keep` or an `exclude`: ask the data owner the cause of a large shift
before a keep; a note expects Monitoring to see it.

In [ ]:
shown = [*views["steps"]["stability"], views["stability_chart"]]
display(*[HTML(block) if isinstance(block, str) else block for block in shown])

## 7. Redundancy
Notes only. `correlated_pair`: two numerics with |Spearman| ≥ 0.90; `associated_pair`: a pair with
a categorical member and Cramér's V or η ≥ 0.90.

**The clusters** come first, one row per group of near-duplicates: `feature` is the member with
the highest concordance, `features` lists every member, `association_min` is the weakest member's
association with that first one and `explained_by_others_max` the most any member is explained by
the others (1: a combination of them, such as a total of its parts). **The table below** has a row
per feature and finding, sorted by finding, then cluster: `value` is the feature's strongest
association, the `detail` its partners with theirs.

A note holds nothing: every member stays a candidate. Where members measure the same thing (a
total beside its parts, one measure over two windows), consider excluding those that add nothing
and keeping the one easiest to explain and to keep filled, often the first. Otherwise leave them:
Training's selection picks among them, and a boosted model splits their importance, so read a
cluster's importances together at the sign-off.

**Decide:** an `exclude` of a near-duplicate that adds nothing, with the cluster as the reason, or
nothing.

In [ ]:
shown = views["steps"]["redundancy"]
display(*[HTML(block) if isinstance(block, str) else block for block in shown])

## 8. Check a finding in the data
Before you decide, look at the rows behind a finding: `snapshot` and `rows` from step 1. Set
`FEATURE` to the feature in question; the examples name the card data's columns.
- **The target by value** (a leak, a rare value, a null set): the snapshot's eligible rows with a
  known target, by the feature's most frequent values, null included. A value with a rate near
  100% is a code the outcome sets.
- **A few clients across the periods** (`client_constant`, `lagged_signal_drop`): one row per
  client, one column per compared period. A static attribute repeats from the start; a look-ahead
  repeats a value it could only know later.
- **The distribution per compared period** (the PSI findings): rows, null share, quartiles and the
  target (the window's periods only) per period. A jump in the null share or the quartiles shows
  the shift. For a numeric only: for a categorical, group `rows` by `period` and `FEATURE`.

`02_eda`'s exploration functions (`pivot_target`, `lift_by_group`, `rate_by_period`) go further
on EDA's result.

In [ ]:
FEATURE = "segment"
labelled = snapshot.loc[(snapshot["eligible"] == 1) & snapshot["target"].notna()]
by_value = labelled.groupby(FEATURE, dropna=False)["target"].agg(["size", "mean"])
by_value.sort_values("size", ascending=False).head(20)

In [ ]:
FEATURE = "onboarding_channel"
clients = snapshot.loc[snapshot["eligible"] == 1, "unit_id"].head(5).tolist()
by_client = rows.where(F.col("unit_id").isin(clients)).select("unit_id", "period", FEATURE)
by_client.toPandas().pivot(index="unit_id", columns="period", values=FEATURE)

In [ ]:
FEATURE = "login_digital_mobile_cnt_0_5_months_avg"
rows.groupBy("period").agg(
    F.count("*").alias("rows"),
    F.avg(F.col(FEATURE).isNull().cast("int")).alias("null_share"),
    F.percentile_approx(FEATURE, [0.25, 0.5, 0.75]).alias("quartiles"),
    F.avg("target").alias("target"),
).orderBy("period").toPandas()

## 9. The proposed decisions and how to record them
A decision is an entry in the decision audit log: append-only, with a reason, applied to every
later run. The table has one row per open flag finding: its threshold and each held feature with
this run's value ("not raised": open from an earlier run). Below it, two calls per finding to copy;
the framework never picks the verb, you do. **Each call takes any subset of the finding's
features:** put each feature in the call you choose (keep some, exclude others, leave the rest
held) and delete the call you do not use.
- **`keep`** settles the findings it names (`findings=[...]`): the feature returns to training,
  and Training's automated selection will not remove it. A different finding raised later holds
  it again; the same finding with new values does not. A feature in two rows stays held until both
  are settled.
- **`exclude`** takes the feature out of training in every later run, whatever its flags, until
  an `include` reverses it, so it needs no `findings`; its `evidence` names the finding it answers,
  for the trail. Feature Checks keeps checking it.
- **Leave it held** when unsure: the model is built without it, and no entry is required.

Paste a call into your own cell (`CONFIG` and `ENVIRONMENT` are the setup cell's), keep the
features it should name and replace `<why>` with a reason a reviewer can check. The CLI does
the same: `python -m core_modelling decide <config> --entity entity_a --keep f1 f2 --findings
<finding> --reason "…"`.

At `assisted`, settle what you can before training. At `autonomous`, record a `keep` for a held
feature you want back, then train again.

In [ ]:
shown = views["proposed_decisions"]
display(*[HTML(block) if isinstance(block, str) else block for block in shown])

**A walk-through on the synthetic data.** The calls below name the card example's synthetic
columns; on other data, copy a call from the table instead. They split one finding's features
between the two verbs and record both entries for real:
- a `keep` of the three static attributes `client_constant` flagged;
- an `exclude` of the fourth, `holding_cnt_trap_lookahead`: one date's value stamped onto every
  period, not known at its period.

The last cell of this section reverts both, so the notebook runs again from the top.

In [ ]:
held_before = result.config.features.held
kept = record_decision(
    CONFIG,
    ENTITY,
    keep=["income", "onboarding_channel", "subsegment"],
    findings=["time_alignment.client_constant"],
    reason="Static attributes set at onboarding, known at every period",
    env_path=ENVIRONMENT,
)
record_decision(
    CONFIG,
    ENTITY,
    exclude=["holding_cnt_trap_lookahead"],
    reason="One date's holdings stamped onto every period: not known at its period",
    evidence=["feature_checks: time_alignment.client_constant"],
    env_path=ENVIRONMENT,
)
show_decisions(cfg, ENTITY, env).tail(3)

**Rerun to see the effect.** The entries apply from the next run: the four features leave the
held set (three kept, one excluded), and the run writes only its `run` entry, since no flag's
features changed. To see the updated views, rerun the display cells above (not the walk-through
cell).

In [ ]:
result = run_feature_checks(cfg, ENTITY, env, spark)
views = result.views
print("no longer held:", sorted(set(held_before) - set(result.config.features.held)))
print("entries written:", [entry["op"] for entry in result.meta["entries_written"]])

**Undo the walk-through.** `revert_to=n` goes back to the state after entry `n`, here the entry
before the keep. The walk-through's entries stay in the log as the trail and are no longer in
force: the four features are held again.

In [ ]:
record_decision(
    CONFIG,
    ENTITY,
    revert_to=kept.seq - 1,
    reason="Undo the notebook walk-through",
    env_path=ENVIRONMENT,
)
held_now = resolve_config(cfg, ENTITY, env).features.held
print("held again:", sorted(set(held_now) - set(result.config.features.held)))

## 10. The report file
`report.html` holds every view above and needs no internet; each run regenerates it. Open the path
the next cell prints in a browser (in JupyterLab: from the file browser, then **Trust HTML**).
Only the report holds:
- **Per period**: every candidate's PSI against the snapshot in each compared period (its
  concordance per window period is in EDA's report).
- **The ledger with decisions**: every candidate with its `verdict` (`pass`, `held`, `kept`,
  `excluded`, or `auto_excluded` for target lineage and unsupported types), the entries in force
  and Feature Checks' statistics.
- **The appendix**: the resolved config, the audit log's entries, the entries this run wrote and
  the timings.

In [ ]:
REPORT = result.paths["report.html"]
print(REPORT)

**Next: `04_training`** (not built yet). At `assisted`, settle the flags you can (step 9), rerun
step 1, then train. At `autonomous`, train at once; Training's log and the model card list the
held features. At both levels a person then signs off every feature of the model before it is
registered; this review is not that sign-off.